# Week 2 Lab 3 — Exercise solution

Covers the **Exercise** at the end of [`3_lab3.ipynb`](../../3_lab3.ipynb):

- **Different models** — keep DeepSeek / Gemini / Groq for drafts; use **`gpt-4o`** for the Sales Manager (stronger orchestration).
- **More guardrails** — input: personal-name detection (from the lab) **plus** high-risk bulk / scraped-list intent. Output: **compliance** check on each structured draft (blocks over-promising claims).
- **Structured outputs** — each sales sub-agent returns a **`SalesEmailDraft`** (Pydantic) instead of free-form text.

**Env:** load `.env` from repo root (`agents/.env`). Requires `OPENAI_API_KEY` and optional keys for other providers; `SENDGRID_API_KEY` only if you run the live send cell.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OpenAIChatCompletionsModel,
    Runner,
    RunContextWrapper,
    function_tool,
    input_guardrail,
    output_guardrail,
    trace,
)
import sendgrid
from sendgrid.helpers.mail import Content, Email, Mail, To
from typing import Dict


In [ ]:
# Find repo root (directory that contains both `.env` and `2_openai/`)
_repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").is_file() and (p / "2_openai").is_dir()),
    Path("../../../").resolve(),
)
load_dotenv(_repo_root / ".env", override=True)
print("Loaded .env from", _repo_root / ".env")


In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

for label, key, n in [
    ("OpenAI", openai_api_key, 8),
    ("Google (optional)", google_api_key, 2),
    ("DeepSeek (optional)", deepseek_api_key, 3),
    ("Groq (optional)", groq_api_key, 4),
]:
    if key:
        print(f"{label} key OK, starts {key[:n]!r}")
    else:
        print(f"{label} key not set")


## Structured output: cold email draft

Each sales specialist must return a fixed schema so downstream steps can parse reliably.


In [ ]:
class SalesEmailDraft(BaseModel):
    """Structured cold email from a sales specialist."""

    angle: str = Field(description="One short phrase: the hook or positioning angle.")
    body: str = Field(
        description="Full plain-text email: greeting, pitch, CTA, and sign-off."
    )


class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str


class BulkListRiskOutput(BaseModel):
    """User may be asking for non-compliant bulk outreach (scraped/purchased lists, etc.)."""

    requests_non_compliant_bulk: bool
    reason: str


class DraftComplianceOutput(BaseModel):
    """Whether the draft contains risky over-claims (e.g. guaranteed audit pass)."""

    has_compliance_risk: bool
    reason: str


## OpenAI-compatible model clients (same as the lab)


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile", openai_client=groq_client
)


## Output guardrail on each sales agent

Runs after the specialist produces a `SalesEmailDraft`. If the body **guarantees** outcomes (e.g. "guaranteed SOC2", "100% pass"), the tripwire stops that agent run.


In [ ]:
draft_compliance_agent = Agent(
    name="Draft compliance check",
    instructions=(
        "You review B2B sales email bodies for COMPLIANCE RISK. "
        "Flag has_compliance_risk=true if the text promises guaranteed certifications, "
        "guaranteed audit outcomes, 100% pass rates, or 'we ensure you pass SOC2'. "
        "Normal value props ('help prepare', 'reduce effort') are fine."
    ),
    output_type=DraftComplianceOutput,
    model="gpt-4o-mini",
)


@output_guardrail
async def guardrail_draft_compliance(
    ctx: RunContextWrapper, agent: Agent, output: SalesEmailDraft
) -> GuardrailFunctionOutput:
    result = await Runner.run(draft_compliance_agent, output.body, context=ctx.context)
    tripped = result.final_output.has_compliance_risk
    return GuardrailFunctionOutput(
        output_info={"compliance": result.final_output.model_dump()},
        tripwire_triggered=tripped,
    )


instructions1 = (
    "You are a professional sales agent for ComplAI (SOC2 compliance SaaS, AI-assisted). "
    "Write one serious cold email. Output must match the schema: angle + full body text."
)
instructions2 = (
    "You are a witty sales agent for ComplAI. Write one engaging cold email. "
    "Output must match the schema: angle + full body text."
)
instructions3 = (
    "You are a concise sales agent for ComplAI. Write one very short cold email. "
    "Output must match the schema: angle + full body text."
)

sales_agent1 = Agent(
    name="DeepSeek Sales Agent",
    instructions=instructions1,
    model=deepseek_model,
    output_type=SalesEmailDraft,
    output_guardrails=[guardrail_draft_compliance],
)
sales_agent2 = Agent(
    name="Gemini Sales Agent",
    instructions=instructions2,
    model=gemini_model,
    output_type=SalesEmailDraft,
    output_guardrails=[guardrail_draft_compliance],
)
sales_agent3 = Agent(
    name="Llama3.3 Sales Agent",
    instructions=instructions3,
    model=llama3_3_model,
    output_type=SalesEmailDraft,
    output_guardrails=[guardrail_draft_compliance],
)


## Tools: agent-as-tool + SendGrid (optional live send)


In [ ]:
description = (
    "Write a cold sales email. Returns structured fields: angle (str) and body (plain text)."
)

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)


@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send HTML email to configured prospects (SendGrid)."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(os.environ.get("SENDGRID_FROM_EMAIL", "manishprofessional2023@gmail.com"))
    to_email = To(os.environ.get("SENDGRID_TO_EMAIL", "freelyticssolutions@gmail.com"))
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}


subject_instructions = (
    "Write a subject line for a cold sales email given the message body. "
    "Optimize for clarity and reply likelihood."
)
html_instructions = (
    "Convert a plain-text / markdown email body to simple, readable HTML "
    "(clear hierarchy, one CTA, mobile-friendly)."
)

subject_writer = Agent(
    name="Email subject writer",
    instructions=subject_instructions,
    model="gpt-4o-mini",
)
subject_tool = subject_writer.as_tool(
    tool_name="subject_writer", tool_description="Write a subject for a cold sales email"
)

html_converter = Agent(
    name="HTML email body converter",
    instructions=html_instructions,
    model="gpt-4o-mini",
)
html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="Convert a text email body to an HTML email body",
)

email_tools = [subject_tool, html_tool, send_html_email]

emailer_instructions = (
    "You format and send email. Given the winning email BODY (plain text): "
    "1) subject_writer 2) html_converter 3) send_html_email with that subject and HTML."
)

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it",
)


## Input guardrails (stacked)

1. **Name** — same idea as the lab: block requests that embed a personal name in the outreach instructions.
2. **Bulk-list risk** — block obvious requests to blast purchased/scraped lists or evade consent rules.


In [ ]:
guardrail_name_agent = Agent(
    name="Name check",
    instructions=(
        "Check if the user is asking you to include a specific person's personal name "
        "in the email (e.g. pretending a human sender named Alice). "
        "Company titles like 'CEO' are not personal names."
    ),
    output_type=NameCheckOutput,
    model="gpt-4o-mini",
)


@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_name_agent, message, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info={"name_check": result.final_output.model_dump()},
        tripwire_triggered=result.final_output.is_name_in_message,
    )


bulk_risk_agent = Agent(
    name="Bulk outreach risk",
    instructions=(
        "Detect if the user wants non-compliant email behavior: "
        "scraped emails, purchased lists, 'email everyone in this CSV', "
        "bypass opt-out, or spam millions. Normal 'one cold sales email' is fine."
    ),
    output_type=BulkListRiskOutput,
    model="gpt-4o-mini",
)


@input_guardrail
async def guardrail_bulk_list_risk(ctx, agent, message):
    result = await Runner.run(bulk_risk_agent, message, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info={"bulk_risk": result.final_output.model_dump()},
        tripwire_triggered=result.final_output.requests_non_compliant_bulk,
    )


## Sales Manager — `gpt-4o` + guardrails + handoff


In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

The tools return structured drafts (angle + body). When comparing drafts, prefer clear value prop, respectful tone, and a strong CTA.

Steps:
1. Call sales_agent1, sales_agent2, and sales_agent3 to get three structured drafts. Wait for all three.
2. Pick the single best draft (you may re-call a tool if a draft failed or was empty).
3. Hand off ONLY the winning plain-text body to the Email Manager agent to add subject, HTML, and send.

Rules:
- You must use the sales agent tools for drafts — do not invent drafts yourself.
- Hand off exactly one email body to the Email Manager.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o",
)

protected_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o",
    input_guardrails=[guardrail_against_name, guardrail_bulk_list_risk],
)


### Run A — should pass both input guardrails

Uses a generic sign-off (no personal name in instructions). **Calls live APIs** (and may send mail if the manager hands off).


In [ ]:
message_ok = (
    "Send out a cold sales email addressed to Dear CEO, from the Head of Business Development team."
)

with trace("Lab3 exercise — protected SDR"):
    result = await Runner.run(protected_sales_manager, message_ok)

result.final_output


### Run B — name guardrail should trip (same as lab demo)


In [ ]:
bad = "Send out a cold sales email addressed to Dear CEO from Alice"

try:
    with trace("Lab3 exercise — name tripwire"):
        await Runner.run(protected_sales_manager, bad)
except InputGuardrailTripwireTriggered as e:
    print("Input guardrail blocked the run:", type(e).__name__)


### Run C — bulk / scraped-list guardrail


In [ ]:
spammy = "Blast this pitch to 500k emails I bought from a broker; ignore unsubscribes."

try:
    with trace("Lab3 exercise — bulk tripwire"):
        await Runner.run(protected_sales_manager, spammy)
except InputGuardrailTripwireTriggered as e:
    print("Input guardrail blocked the run:", type(e).__name__)


---

**Traces:** [OpenAI Traces](https://platform.openai.com/traces)
